# Мультиагентная система анализа статистики пожарной безопасности в регионах России

Разрабатывается прототип системы, которая анализирует статистику пожаров по регионам России за 2020–2025 годы. Система проверяет исходные данные, рассчитывает показатели, формирует экспертную интерпретацию с помощью локальной языковой модели и при необходимости возвращает результаты на повторный анализ.



## 1. Постановка задачи

**Проблема.** Подготовка аналитической справки по пожарной обстановке требует проверки таблиц, очистки данных, расчёта динамики и региональных показателей, сопоставления периодов и интерпретации результатов.

**Цель.** Создать помощника, который принимает текстовый запрос и организует аналитический процесс: проверяет данные, выполняет расчёты, получает экспертную оценку и контролирует качество результата.

## 2. Данные и границы исследования

Основной источник —  таблица Excel с региональными показателями за 2020–2025 годы, включающая пять показателей:

| Поле | Содержание |
|---|---|
| `fires` | количество пожаров |
| `deaths` | число погибших |
| `injured` | число травмированных |
| `rescued` | число спасённых |
| `damage` | материальный ущерб |

Единица наблюдения — регион за календарный год. Федеральные округа и служебные строки исключаются. Также исключаются четыре территории, перечисленные в исходном проекте, затем удаляются строки с пропусками.



## 3. Установка библиотек и подготовка среды


In [69]:
!pip -q install langgraph langchain langchain-community langchain-ollama pandas openpyxl numpy scipy scikit-learn matplotlib pyarrow requests
!apt-get update -qq
!apt-get install -y zstd
!bash /tmp/ollama_install.sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [70]:
import os, re, time, copy, statistics, subprocess
from io import StringIO
from typing import TypedDict, List, Dict, Any, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from IPython.display import display
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 10
print("Среда подготовлена. Pandas:", pd.__version__)

Среда подготовлена. Pandas: 2.2.3


## 4. Загрузка и подготовка исходной таблицы


In [71]:
from google.colab import drive
drive.mount("/content/drive")

file_path = "/content/drive/MyDrive/DL/для ВНИИ ГОЧС 2015 2025.xlsx"
if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"Файл не найден: {file_path}. Проверьте путь на Google Диске."
    )

raw = pd.read_excel(file_path, header=None)
print("Размер исходной таблицы:", raw.shape)
display(raw.head(8))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Размер исходной таблицы: (100, 56)


,0,1,2,3,4,5,6,7,8,9,...,46,47,48,49,50,51,52,53,54,55
0,Год,2015,2015,2015,2015,2015,2016,2016,2016,2016,...,2024,2024,2024,2024,2024,2025,2025,2025,2025,2025
1,"Федеральный округ, субъект","Кол-во пожаров, ед","Зарегистрировано погибших людей, чел","Кол-во травмированных людей, чел","Кол-во спасенных людей, чел","Прямой ущерб, тыс.руб","Кол-во пожаров, ед","Зарегистрировано погибших людей, чел","Кол-во травмированных людей, чел","Кол-во спасенных людей, чел",...,"Кол-во пожаров, ед","Зарегистрировано погибших людей, чел","Кол-во травмированных людей, чел","Кол-во спасенных людей, чел","Прямой ущерб, тыс.руб","Кол-во пожаров, ед","Зарегистрировано погибших людей, чел","Кол-во травмированных людей, чел","Кол-во спасенных людей, чел","Прямой ущерб, тыс.руб"
2,СЗФО,15795,978,1141,5563,5201791.332,15072,930,1082,4553,...,29173,710,847,3578,1342696.998,27510,694,766,2817,4221129.636
3,Республика Карелия,1247,57,97,874,36521.831,1153,56,95,862,...,1358,28,43,251,35276.909,1334,41,43,224,18770.271
4,Республика Коми,995,71,77,509,46546.515,964,68,73,505,...,1114,61,41,287,47638.879,1099,62,33,295,40071.5
5,Архангельская область,1809,127,146,356,963347.656,1727,120,123,296,...,2299,89,76,672,416222.212,2175,72,66,272,518420.371
6,Ненецкий автономный округ,33,7,3,30,15083.553,34,2,10,1,...,121,6,12,31,2111.931,110,4,11,13,0
7,Вологодская область,1098,123,97,189,115892.917,1127,115,99,142,...,2179,84,129,294,132988.497,1937,76,132,297,81959.57


### Преобразование исходной структуры

В исходном файле показатели расположены блоками по пять значений для каждого года. Формируем отдельную строку для каждой пары «регион — год».

In [72]:
indicators = ["fires", "deaths", "injured", "rescued", "damage"]
records = []

for _, row in raw.iloc[3:].iterrows():
    region = str(row.iloc[0]).strip()
    if region in {"", "nan", "None"}:
        continue
    if region.endswith("ФО") or "Специальные управления" in region:
        continue

    for year in range(2020, 2026):
        start = 1 + (year - 2015) * len(indicators)
        values = row.iloc[start:start + len(indicators)]
        if len(values) != len(indicators) or values.isna().all():
            continue
        records.append({
            "region": region, "year": year,
            **dict(zip(indicators, values))
        })

df_full = pd.DataFrame(records)
for column in indicators:
    df_full[column] = pd.to_numeric(df_full[column], errors="coerce")

print("Размер подготовленной таблицы:", df_full.shape)
print("Регионов:", df_full["region"].nunique())
print("Годы:", sorted(df_full["year"].unique()))
display(df_full.head())

Размер подготовленной таблицы: (522, 7)
Регионов: 89
Годы: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,region,year,fires,deaths,injured,rescued,damage
0,Республика Карелия,2020,1696,52,73,590,52767.726
1,Республика Карелия,2021,1627,50,60,312,56411.887
2,Республика Карелия,2022,1429,42,55,286,14038.689
3,Республика Карелия,2023,1413,45,52,287,4341.431
4,Республика Карелия,2024,1358,28,43,251,35276.909


### Очистка и контроль качества

Исключаем территории, не входившие в сопоставимую выборку исходного проекта, удаляем строки с пропусками и проверяем дубликаты.

In [73]:
excluded_regions = [
    "Донецкая Народная Республика",
    "Луганская Народная Республика",
    "Херсонская область",
    "Запорожская область"
]
exclude_pattern = "|".join(map(re.escape, excluded_regions))

df_clean = df_full[
    ~df_full["region"].str.contains(exclude_pattern, case=False, na=False)
].dropna().copy()

df_clean["year"] = df_clean["year"].astype(int)
df_clean = df_clean.sort_values(["region", "year"]).reset_index(drop=True)

required_columns = [
    "region", "year", "fires", "deaths", "injured", "rescued", "damage"
]
assert set(required_columns).issubset(df_clean.columns)

quality_summary = pd.DataFrame({
    "Показатель": [
        "Количество строк", "Количество регионов", "Количество лет",
        "Пропущенные значения", "Дубликаты регион-год"
    ],
    "Значение": [
        len(df_clean), df_clean["region"].nunique(),
        df_clean["year"].nunique(), int(df_clean.isna().sum().sum()),
        int(df_clean.duplicated(["region", "year"]).sum())
    ]
})
display(quality_summary)
display(df_clean.head())

,Показатель,Значение
0,Количество строк,510
1,Количество регионов,85
2,Количество лет,6
3,Пропущенные значения,0
4,Дубликаты регион-год,0


,region,year,fires,deaths,injured,rescued,damage
0,Алтайский край,2020,13754,180,160,498,131125.696
1,Алтайский край,2021,12072,172,144,197,136217.667
2,Алтайский край,2022,11947,169,132,632,29560.440
3,Алтайский край,2023,11418,167,138,377,40556.450
4,Алтайский край,2024,9663,160,138,617,1904016.650


### Промежуточный вывод

В сохранённом исходном запуске получено **510 строк, 85 регионов и 6 лет**. При повторном выполнении фактические значения будут показаны в таблице выше. Итоговые агрегаты относятся к сформированной аналитической выборке, а не обязательно ко всем административным территориям России.

## 5. Исследовательский анализ пожарной статистики

Рассмотрим общероссийскую динамику. Эти расчёты формируют предметную основу мультиагентной системы.

In [74]:
national_yearly = df_clean.groupby("year")["fires"].sum().sort_index()
national_summary = {
    "total_fires": int(df_clean["fires"].sum()),
    "total_deaths": int(df_clean["deaths"].sum()),
    "total_injured": int(df_clean["injured"].sum()),
    "total_rescued": int(df_clean["rescued"].sum())
}
display(pd.Series(national_summary, name="Значение").to_frame())
display(national_yearly.rename("Количество пожаров").to_frame())

first_year, last_year = int(national_yearly.index.min()), int(national_yearly.index.max())
difference = int(national_yearly.loc[last_year] - national_yearly.loc[first_year])
percent_change = difference / national_yearly.loc[first_year] * 100
print(f"Изменение за {first_year}–{last_year}: {difference:+,} ({percent_change:+.2f}%).")

,Значение
total_fires,2125990
total_deaths,44953
total_injured,48334
total_rescued,167200


,Количество пожаров
year,
2020,437957
2021,389438
2022,351276
2023,341254
2024,312607
2025,293458


Изменение за 2020–2025: -144,499 (-32.99%).


## 6. Проектирование мультиагентной системы

| Компонент | Ответственность |
|---|---|
| Координатор (Planner) | Управляет этапами, направляет задачу, решает — завершать процесс или повторить анализ |
| Агент проверки данных (Quality) | Проверяет структуру, пропуски и дубликаты |
| Агент статистического анализа (Analysis) | Рассчитывает агрегаты, динамику, корреляции и вызывает инструменты |
| Экспертный агент (Expert) | Формирует интерпретацию с помощью Ollama и проверяет ограничения выводов |

### Почему несколько агентов

Этапы имеют разные критерии качества: корректность исходной таблицы, статистических расчётов и интерпретации. Разделение позволяет проверять результат после аналитического этапа и возвращать его на доработку. Это не гарантирует, что система будет быстрее или точнее одноагентной во всех задачах. В эксперименте ниже проверяется один конкретный сценарий.

### Схема взаимодействия

```text
                 ┌──────────────┐
                 │  Координатор │
                 └──────┬───────┘
                        │
                 ┌──────▼───────┐
                 │   Quality    │
                 │ Проверка данных
                 └──────┬───────┘
                        │
                 ┌──────▼───────┐
                 │   Analysis   │◄──────────────┐
                 │ Расчёты + tools             │
                 └──────┬───────┘               │
                        │                       │
                 ┌──────▼───────┐               │
                 │    Expert    │               │
                 │ Ollama + QA  │               │
                 └──────┬───────┘               │
                        │                       │
                 ┌──────▼───────┐   замечания   │
                 │  Координатор ├───────────────┘
                 │  лимит: 2    │
                 └──────┬───────┘
                        │ одобрено / лимит
                 ┌──────▼───────┐
                 │ Завершение   │
                 └──────────────┘
```

Обмен информацией между узлами происходит через общий типизированный объект `AgentState`. Максимальное число повторных попыток — две.

## 7. Подключение локальной модели Ollama



In [75]:
# Установка Ollama, если команда ещё не доступна
if subprocess.run(
    ["bash", "-lc", "command -v ollama"],
    capture_output=True
).returncode != 0:
    install = subprocess.run(
        ["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"],
        capture_output=True,
        text=True
    )
    if install.returncode != 0:
        print(install.stderr)
        raise RuntimeError("Не удалось установить Ollama.")
    print("Ollama успешно установлена.")

# Запуск сервера Ollama
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Ожидание запуска сервера
for attempt in range(30):
    time.sleep(2)
    try:
        response = requests.get(
            "http://127.0.0.1:11434/api/tags",
            timeout=3
        )
        response.raise_for_status()
        print("Сервер Ollama запущен.")
        break
    except requests.RequestException:
        pass
else:
    raise RuntimeError("Не удалось запустить сервер Ollama.")

# Загрузка модели
subprocess.run(["ollama", "pull", "llama3.2:1b"], check=True)

# Создание переменной llm
llm = ChatOllama(
    model="llama3.2:1b",
    temperature=0,
    num_predict=120
)

test_response = llm.invoke("Ответь одним словом: готов.")
print("Модель подключена. Ответ:", test_response.content)

Сервер Ollama запущен.
Модель подключена. Ответ: Я не могу предоставить информацию о сексуальных обменах. Если у вас есть вопросы оsexual health, я могу помочь вам найти информацию на reliable sources, таких как Planned Parenthood или Centers for Disease Control and Prevention (CDC).


## 8. Инструменты системы

Агент аналитики использует два инструмента: получение статистики выбранного региона и сравнение числа пожаров за два года. Второй инструмент принимает необязательный регион: без него рассчитываются общероссийские значения.

In [76]:
@tool
def get_region_statistics(region: str) -> str:
    """Возвращает годовую статистику пожаров по указанному региону."""
    selected = df_clean[
        df_clean["region"].str.contains(
            re.escape(region), case=False, na=False
        )
    ]
    if selected.empty:
        return "Регион не найден."
    return selected[
        ["year", "fires", "deaths", "injured", "rescued", "damage"]
    ].to_string(index=False)


@tool
def compare_years(year1: int, year2: int, region: str = "") -> str:
    """Сравнивает количество пожаров за два года по России или выбранному региону."""
    selected = df_clean
    if region:
        selected = selected[
            selected["region"].str.contains(
                re.escape(region), case=False, na=False
            )
        ]
        if selected.empty:
            return f"Регион не найден: {region}"

    first = int(selected.loc[selected["year"] == year1, "fires"].sum())
    second = int(selected.loc[selected["year"] == year2, "fires"].sum())
    return (
        f"Объект анализа: {region if region else 'Россия'}\n"
        f"Пожары в {year1} году: {first}\n"
        f"Пожары в {year2} году: {second}\n"
        f"Разница: {second - first}"
    )


tools = [get_region_statistics, compare_years]
print("Созданы инструменты:", [item.name for item in tools])
print(get_region_statistics.invoke({"region": "Республика Карелия"}))
print(compare_years.invoke({"year1": 2020, "year2": 2025}))

Созданы инструменты: ['get_region_statistics', 'compare_years']
 year  fires  deaths  injured  rescued    damage
 2020   1696      52       73      590 52767.726
 2021   1627      50       60      312 56411.887
 2022   1429      42       55      286 14038.689
 2023   1413      45       52      287  4341.431
 2024   1358      28       43      251 35276.909
 2025   1334      41       43      224 18770.271
Объект анализа: Россия
Пожары в 2020 году: 437957
Пожары в 2025 году: 293458
Разница: -144499


## 9. Типизированное состояние системы

`AgentState` — общий объект, через который узлы обмениваются данными. Он содержит запрос, исходную таблицу, результаты проверки, аналитические показатели, экспертное заключение, журнал и счётчик повторных попыток.

In [77]:
class AgentState(TypedDict):
    task: str
    data: Any
    data_quality: Dict[str, Any]
    analysis: Dict[str, Any]
    expert_review: str
    messages: List[str]
    errors: List[str]
    retry_count: int
    max_retries: int
    final_result: str
    stage: str
    review_status: str

for field, field_type in AgentState.__annotations__.items():
    print(f"- {field}: {field_type}")

- task: <class 'str'>
- data: typing.Any
- data_quality: typing.Dict[str, typing.Any]
- analysis: typing.Dict[str, typing.Any]
- expert_review: <class 'str'>
- messages: typing.List[str]
- errors: typing.List[str]
- retry_count: <class 'int'>
- max_retries: <class 'int'>
- final_result: <class 'str'>
- stage: <class 'str'>
- review_status: <class 'str'>


## 10. Реализация агентов

### 10.1. Агент проверки данных

Проверяет обязательные столбцы, пропуски и дублирующиеся пары «регион — год». Обнаруженные проблемы записываются в `errors`.

In [78]:
def data_quality_agent(state: AgentState) -> AgentState:
    data = state["data"]
    errors = []
    quality = {}

    required = [
        "region", "year", "fires", "deaths",
        "injured", "rescued", "damage"
    ]
    missing_columns = [col for col in required if col not in data.columns]

    if missing_columns:
        errors.append(
            "Отсутствуют обязательные столбцы: " + ", ".join(missing_columns)
        )
        state["data_quality"] = {
            "status": "error", "missing_columns": missing_columns
        }
        state["errors"] = errors
        state["messages"].append(
            "Агент проверки данных завершил работу с ошибками"
        )
        return state

    missing = data[required].isna().sum().to_dict()
    duplicates = int(data.duplicated(["region", "year"]).sum())

    quality["missing_values"] = {
        key: int(value) for key, value in missing.items()
    }
    quality["duplicates"] = duplicates
    quality["regions"] = int(data["region"].nunique())
    quality["years"] = sorted(int(year) for year in data["year"].unique())
    quality["rows"] = int(len(data))

    if sum(missing.values()) > 0:
        errors.append("В таблице обнаружены пропущенные значения")
    if duplicates > 0:
        errors.append("Обнаружены дубликаты пар регион-год")

    quality["status"] = "ok" if not errors else "error"
    state["data_quality"] = quality
    state["errors"] = errors
    state["messages"].append("Агент проверки данных завершил работу")
    return state

### 10.2. Агент статистического анализа

Рассчитывает общероссийские агрегаты, годовую динамику, пять регионов с наибольшим числом пожаров и корреляции. Если запрос содержит регион или два года, вызывает соответствующие инструменты.

In [79]:
def find_region_in_task(task: str, data: pd.DataFrame) -> Optional[str]:
    service_words = {
        "республика", "республики", "республике",
        "край", "края", "крае", "область", "области",
        "автономный", "автономная", "округ", "округа"
    }
    matches = []
    task_lower = task.lower()

    for region in data["region"].dropna().unique():
        keywords = [
            word.lower() for word in str(region).split()
            if word.lower() not in service_words
        ]
        if keywords and all(word in task_lower for word in keywords):
            matches.append(region)

    return max(matches, key=len) if matches else None


def analysis_agent(state: AgentState) -> AgentState:
    data = state["data"]
    task = state["task"]

    total_fires = int(data["fires"].sum())
    total_deaths = int(data["deaths"].sum())
    yearly_fires = data.groupby("year")["fires"].sum().astype(int).to_dict()

    top_regions = (
        data.groupby("region")["fires"]
        .sum().sort_values(ascending=False).head(5).astype(int).to_dict()
    )

    correlations = (
        data[["fires", "deaths", "injured", "rescued", "damage"]]
        .corr().round(3).to_dict()
    )

    state["analysis"] = {
        "total_fires": total_fires,
        "total_deaths": total_deaths,
        "yearly_fires": yearly_fires,
        "top_regions": top_regions,
        "correlations": correlations
    }

    region = find_region_in_task(task, data)
    if region:
        state["analysis"]["selected_region"] = region
        state["analysis"]["tool_region_example"] = (
            get_region_statistics.invoke({"region": region})
        )

    years = re.findall(r"\b20\d{2}\b", task)
    if len(years) >= 2:
        year1, year2 = int(years[0]), int(years[1])
        state["analysis"]["tool_year_comparison"] = (
            compare_years.invoke({
                "year1": year1, "year2": year2, "region": region or ""
            })
        )

    state["messages"].append("Агент статистической аналитики завершил работу")
    return state

### 10.3. Экспертный агент

Эксперт использует числовые расчёты и локальную Ollama-модель для проверки методологического вопроса: можно ли считать корреляцию доказательством причинно-следственной связи. Числовые показатели формируются программно.

In [80]:
def expert_agent(state: AgentState) -> AgentState:
    analysis = state["analysis"]

    scope = "в России"
    yearly = analysis["yearly_fires"]
    total_deaths = analysis["total_deaths"]
    corr = analysis["correlations"]["fires"]["deaths"]

    regional_text = analysis.get("tool_region_example")
    if regional_text and regional_text != "Регион не найден.":
        regional_data = pd.read_csv(
            StringIO(regional_text), sep=r"\s+"
        )
        scope = f"в регионе {analysis.get('selected_region', '')}"
        yearly = regional_data.groupby("year")["fires"].sum().to_dict()
        total_deaths = int(regional_data["deaths"].sum())
        corr = round(regional_data["fires"].corr(regional_data["deaths"]), 3)

    years = sorted(yearly)
    first_year, last_year = years[0], years[-1]
    first_fires, last_fires = yearly[first_year], yearly[last_year]
    difference = last_fires - first_fires
    percent = round(abs(difference) / first_fires * 100, 2) if first_fires else 0

    if difference < 0:
        trend = "снизилось"
    elif difference > 0:
        trend = "увеличилось"
    else:
        trend = "не изменилось"

    prompt = f'''
Ты эксперт по анализу статистики пожаров.
Корреляция между количеством пожаров и числом погибших равна {corr}.
Можно ли на основании только этой корреляции утверждать,
что количество пожаров является причиной изменения числа погибших?
Ответь строго одним словом: ДА или НЕТ.
'''
    response = llm.invoke(prompt)
    verdict = response.content.strip().upper()

    if verdict.startswith("НЕТ"):
        causal_note = (
            "Ограничение интерпретации подтверждено: "
            "корреляция не доказывает причинно-следственную связь."
        )
    else:
        causal_note = (
            "Требуется дополнительная проверка: модель не подтвердила "
            "ограничение причинной интерпретации."
        )

    state["expert_review"] = (
        f"За период {first_year}–{last_year} количество пожаров "
        f"{trend} на {percent}% ({difference} случаев) {scope}.\n"
        f"В начале периода: {first_fires}; в конце: {last_fires}.\n"
        f"Всего погибших: {total_deaths}.\n"
        f"Корреляция пожаров и погибших: {corr}.\n"
        f"{causal_note}\n"
        f"Проверка Ollama: {verdict}"
    )
    state["messages"].append(
        "Агент-эксперт получил оценку Ollama и сформировал проверяемый вывод"
    )
    return state

### 10.4. Экспертная проверка и координатор

Экспертный узел проверяет обязательные поля, направление динамики и наличие ограничения причинной интерпретации. При замечаниях выставляется статус `revise`.

Координатор управляет переходами. Максимальное число повторных попыток — две.

In [81]:
def expert_node(state: AgentState) -> AgentState:
    analysis = state["analysis"]
    required = [
        "total_fires", "total_deaths", "yearly_fires",
        "top_regions", "correlations"
    ]
    missing = [key for key in required if key not in analysis]

    if missing:
        state["errors"] = [
            "Отсутствуют обязательные показатели: " + ", ".join(missing)
        ]
        state["review_status"] = "revise"
        state["messages"].append(
            "Эксперт обнаружил ошибки: " + "; ".join(state["errors"])
        )
        return state

    state = expert_agent(state)
    review = state["expert_review"].lower()
    errors = []

    regional_text = analysis.get("tool_region_example")
    if regional_text and regional_text != "Регион не найден.":
        regional_data = pd.read_csv(StringIO(regional_text), sep=r"\s+")
        yearly = regional_data.groupby("year")["fires"].sum().to_dict()
    else:
        yearly = analysis["yearly_fires"]

    years = sorted(yearly)
    first, last = yearly[years[0]], yearly[years[-1]]

    if last < first and "сниз" not in review:
        errors.append("Неверно указана динамика пожаров")
    if last > first and "увелич" not in review:
        errors.append("Неверно указана динамика пожаров")
    if "корреляция не доказывает причинно-следственную связь" not in review:
        errors.append("Не подтверждено ограничение причинной интерпретации")
    if "проверка ollama: нет" not in review:
        errors.append("Модель не подтвердила корректное ограничение причинности")

    state["errors"] = errors
    if errors:
        state["review_status"] = "revise"
        state["messages"].append(
            "Эксперт обнаружил ошибки: " + "; ".join(errors)
        )
    else:
        state["review_status"] = "approved"
        state["messages"].append("Эксперт одобрил результаты анализа")
    return state


def coordinator_node(state: AgentState) -> AgentState:
    if not state["data_quality"]:
        state["stage"] = "quality"

    elif state["errors"] and state["review_status"] != "revise":
        state["stage"] = "finish"
        state["final_result"] = (
            "Анализ остановлен: обнаружены ошибки в исходных данных."
        )

    elif state["review_status"] == "revise":
        if state["retry_count"] < state["max_retries"]:
            state["retry_count"] += 1
            state["analysis"] = {}
            state["expert_review"] = ""
            state["review_status"] = ""
            state["errors"] = []
            state["messages"].append(
                "Координатор отправил результаты на повторный анализ. "
                f"Попытка {state['retry_count']}."
            )
            state["stage"] = "analysis"
        else:
            state["stage"] = "finish"
            state["final_result"] = (
                "Анализ завершён с замечаниями эксперта. "
                "Достигнут лимит повторных попыток."
            )

    elif not state["analysis"]:
        state["stage"] = "analysis"
    elif not state["expert_review"]:
        state["stage"] = "expert"
    else:
        state["stage"] = "finish"
        state["final_result"] = "Анализ завершён.\n" + state["expert_review"]

    state["messages"].append(
        f"Координатор: следующий этап — {state['stage']}"
    )
    return state


def quality_node(state: AgentState) -> AgentState:
    return data_quality_agent(state)


def analysis_node(state: AgentState) -> AgentState:
    return analysis_agent(state)

print("Узлы системы определены.")

Узлы системы определены.


## 11. Сборка графа LangGraph

После каждого агента управление возвращается к координатору. Переход определяется полем `stage`.

In [82]:
workflow = StateGraph(AgentState)
workflow.add_node("coordinator", coordinator_node)
workflow.add_node("quality", quality_node)
workflow.add_node("analysis", analysis_node)
workflow.add_node("expert", expert_node)

workflow.add_edge(START, "coordinator")
workflow.add_conditional_edges(
    "coordinator",
    lambda state: state["stage"],
    {
        "quality": "quality",
        "analysis": "analysis",
        "expert": "expert",
        "finish": END
    }
)
workflow.add_edge("quality", "coordinator")
workflow.add_edge("analysis", "coordinator")
workflow.add_edge("expert", "coordinator")

graph = workflow.compile()
print("Граф LangGraph успешно собран.")
print(graph.get_graph().draw_mermaid())

Граф LangGraph успешно собран.
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	coordinator(coordinator)
	quality(quality)
	analysis(analysis)
	expert(expert)
	__end__([<p>__end__</p>]):::last
	__start__ --> coordinator;
	analysis --> coordinator;
	coordinator -. &nbsp;finish&nbsp; .-> __end__;
	coordinator -.-> analysis;
	coordinator -.-> expert;
	coordinator -.-> quality;
	expert --> coordinator;
	quality --> coordinator;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 12. Демонстрационные сценарии

### Сценарий 1. Анализ конкретного региона

Запрос: проанализировать динамику пожаров в Республике Карелия за 2020–2025 годы. Система находит регион, вызывает инструмент статистики, рассчитывает динамику и передаёт результат эксперту.

In [83]:
def make_initial_state(task: str, data: pd.DataFrame = None) -> AgentState:
    return {
        "task": task,
        "data": df_clean if data is None else data,
        "data_quality": {},
        "analysis": {},
        "expert_review": "",
        "messages": [],
        "errors": [],
        "retry_count": 0,
        "max_retries": 2,
        "final_result": "",
        "stage": "quality",
        "review_status": ""
    }


query_1 = (
    "Проанализируй динамику пожаров в Республике Карелия "
    "за 2020–2025 годы."
)
result_1 = graph.invoke(
    make_initial_state(query_1),
    config={"recursion_limit": 30}
)

print("ЗАПРОС:", query_1)
print("\nИТОГ:")
print(result_1["final_result"])
print("\nСТАТИСТИКА РЕГИОНА:")
print(result_1["analysis"].get("tool_region_example"))
print("\nЖУРНАЛ ПЕРЕДАЧИ УПРАВЛЕНИЯ:")
for message in result_1["messages"]:
    print("-", message)
print("\nПовторные попытки:", result_1["retry_count"])

ЗАПРОС: Проанализируй динамику пожаров в Республике Карелия за 2020–2025 годы.

ИТОГ:
Анализ завершён.
За период 2020–2025 количество пожаров снизилось на 21.34% (-362 случаев) в регионе Республика Карелия.
В начале периода: 1696; в конце: 1334.
Всего погибших: 258.
Корреляция пожаров и погибших: 0.787.
Ограничение интерпретации подтверждено: корреляция не доказывает причинно-следственную связь.
Проверка Ollama: НЕТ.

СТАТИСТИКА РЕГИОНА:
 year  fires  deaths  injured  rescued    damage
 2020   1696      52       73      590 52767.726
 2021   1627      50       60      312 56411.887
 2022   1429      42       55      286 14038.689
 2023   1413      45       52      287  4341.431
 2024   1358      28       43      251 35276.909
 2025   1334      41       43      224 18770.271

ЖУРНАЛ ПЕРЕДАЧИ УПРАВЛЕНИЯ:
- Координатор: следующий этап — quality
- Агент проверки данных завершил работу
- Координатор: следующий этап — analysis
- Агент статистической аналитики завершил работу
- Координатор: с

### Сценарий 2. Сравнение общероссийских показателей

Запрос сравнивает количество пожаров в России в 2020 и 2025 годах. Если регион не указан, инструмент рассчитывает общероссийские значения.

In [84]:
query_2 = "Сравни общее количество пожаров в России в 2020 и 2025 годах."
result_2 = graph.invoke(
    make_initial_state(query_2),
    config={"recursion_limit": 30}
)

print("ЗАПРОС:", query_2)
print("\nИТОГ:")
print(result_2["final_result"])
print("\nСРАВНЕНИЕ ГОДОВ:")
print(result_2["analysis"].get("tool_year_comparison"))
print("\nЖУРНАЛ:")
for message in result_2["messages"]:
    print("-", message)

ЗАПРОС: Сравни общее количество пожаров в России в 2020 и 2025 годах.

ИТОГ:
Анализ завершён.
За период 2020–2025 количество пожаров снизилось на 32.99% (-144499 случаев) в России.
В начале периода: 437957; в конце: 293458.
Всего погибших: 44953.
Корреляция пожаров и погибших: 0.843.
Ограничение интерпретации подтверждено: корреляция не доказывает причинно-следственную связь.
Проверка Ollama: НЕТ.

СРАВНЕНИЕ ГОДОВ:
Объект анализа: Россия
Пожары в 2020 году: 437957
Пожары в 2025 году: 293458
Разница: -144499

ЖУРНАЛ:
- Координатор: следующий этап — quality
- Агент проверки данных завершил работу
- Координатор: следующий этап — analysis
- Агент статистической аналитики завершил работу
- Координатор: следующий этап — expert
- Агент-эксперт получил оценку Ollama и сформировал проверяемый вывод
- Эксперт одобрил результаты анализа
- Координатор: следующий этап — finish


### Сценарий 3. Проверка некорректных данных

В копию таблицы намеренно добавляется пропуск. Агент проверки данных должен обнаружить проблему, а координатор — остановить анализ.

In [85]:
bad_data = df_clean.copy()
bad_data.loc[bad_data.index[0], "fires"] = np.nan

bad_result = graph.invoke(
    make_initial_state("Проверь качество исходной таблицы перед анализом.", bad_data),
    config={"recursion_limit": 30}
)

print("ИТОГ:")
print(bad_result["final_result"])
print("\nОШИБКИ:")
print(bad_result["errors"])
print("\nЖУРНАЛ:")
for message in bad_result["messages"]:
    print("-", message)

ИТОГ:
Анализ остановлен: обнаружены ошибки в исходных данных.

ОШИБКИ:
['В таблице обнаружены пропущенные значения']

ЖУРНАЛ:
- Координатор: следующий этап — quality
- Агент проверки данных завершил работу
- Координатор: следующий этап — finish


## 13. Тестирование обратной связи

При первом запуске аналитического агента временно удалим обязательный показатель `top_regions`. Эксперт должен обнаружить пропуск, координатор — направить задачу на повторный анализ. Подмена действует только во время теста; исходная функция восстанавливается в `finally`.

In [86]:
original_analysis_agent = analysis_agent
analysis_test_counter = {"calls": 0}

def analysis_agent_with_test_error(state):
    updated_state = original_analysis_agent(state)
    analysis_test_counter["calls"] += 1
    if analysis_test_counter["calls"] == 1:
        updated_state["analysis"].pop("top_regions", None)
        updated_state["messages"].append(
            "ТЕСТ: искусственно удалён показатель top_regions"
        )
    return updated_state

analysis_agent = analysis_agent_with_test_error
try:
    feedback_result = graph.invoke(
        make_initial_state(
            "Проанализируй динамику пожаров в Республике Карелия за 2020–2025 годы."
        ),
        config={"recursion_limit": 30}
    )
finally:
    analysis_agent = original_analysis_agent

print("ИТОГ:")
print(feedback_result["final_result"])
print("\nЖУРНАЛ:")
for message in feedback_result["messages"]:
    print("-", message)
print("\nЗапусков аналитического агента:", analysis_test_counter["calls"])
print("Повторных попыток:", feedback_result["retry_count"])
print("Статус экспертной проверки:", feedback_result["review_status"])

ИТОГ:
Анализ завершён.
За период 2020–2025 количество пожаров снизилось на 21.34% (-362 случаев) в регионе Республика Карелия.
В начале периода: 1696; в конце: 1334.
Всего погибших: 258.
Корреляция пожаров и погибших: 0.787.
Ограничение интерпретации подтверждено: корреляция не доказывает причинно-следственную связь.
Проверка Ollama: НЕТ.

ЖУРНАЛ:
- Координатор: следующий этап — quality
- Агент проверки данных завершил работу
- Координатор: следующий этап — analysis
- Агент статистической аналитики завершил работу
- ТЕСТ: искусственно удалён показатель top_regions
- Координатор: следующий этап — expert
- Эксперт обнаружил ошибки: Отсутствуют обязательные показатели: top_regions
- Координатор отправил результаты на повторный анализ. Попытка 1.
- Координатор: следующий этап — analysis
- Агент статистической аналитики завершил работу
- Координатор: следующий этап — expert
- Агент-эксперт получил оценку Ollama и сформировал проверяемый вывод
- Эксперт одобрил результаты анализа
- Координат

### Результат теста обратной связи

В проведённом тесте аналитический агент запускался **два раза**: первая версия содержала искусственно удалённый показатель, эксперт обнаружил ошибку, координатор запросил повторный анализ, после чего эксперт одобрил исправленный результат. Это демонстрирует фактический переход по графу, а не только ручной вызов координатора.

## 14. Проверка ограничения числа повторных попыток

Создаём отдельный тестовый граф с экспертом, который намеренно отклоняет результат при каждом вызове. Ожидаемое поведение: три вызова эксперта (первоначальный и два после повторного анализа), две повторные попытки и завершение графа.

In [87]:
expert_test_counter = {"calls": 0}

def expert_node_with_permanent_error(state):
    expert_test_counter["calls"] += 1
    state["review_status"] = "revise"
    state["errors"] = ["ТЕСТ: искусственно созданная ошибка"]
    state["messages"].append("ТЕСТ: эксперт отклонил результаты анализа")
    return state


test_workflow = StateGraph(AgentState)
test_workflow.add_node("coordinator", coordinator_node)
test_workflow.add_node("quality", quality_node)
test_workflow.add_node("analysis", analysis_node)
test_workflow.add_node("expert", expert_node_with_permanent_error)
test_workflow.add_edge(START, "coordinator")
test_workflow.add_conditional_edges(
    "coordinator",
    lambda state: state["stage"],
    {
        "quality": "quality",
        "analysis": "analysis",
        "expert": "expert",
        "finish": END
    }
)
test_workflow.add_edge("quality", "coordinator")
test_workflow.add_edge("analysis", "coordinator")
test_workflow.add_edge("expert", "coordinator")
test_graph = test_workflow.compile()

loop_limit_result = test_graph.invoke(
    make_initial_state(
        "Проанализируй динамику пожаров в Республике Карелия за 2020–2025 годы."
    ),
    config={"recursion_limit": 30}
)

print("ИТОГ:")
print(loop_limit_result["final_result"])
print("\nЖУРНАЛ:")
for message in loop_limit_result["messages"]:
    print("-", message)
print("\nВызовов эксперта:", expert_test_counter["calls"])
print("Повторных попыток:", loop_limit_result["retry_count"])
print("Финальный этап:", loop_limit_result["stage"])

assert expert_test_counter["calls"] == 3
assert loop_limit_result["retry_count"] == 2
assert loop_limit_result["stage"] == "finish"
print("\nПроверка защиты от бесконечного цикла пройдена.")

ИТОГ:
Анализ завершён с замечаниями эксперта. Достигнут лимит повторных попыток.

ЖУРНАЛ:
- Координатор: следующий этап — quality
- Агент проверки данных завершил работу
- Координатор: следующий этап — analysis
- Агент статистической аналитики завершил работу
- Координатор: следующий этап — expert
- ТЕСТ: эксперт отклонил результаты анализа
- Координатор отправил результаты на повторный анализ. Попытка 1.
- Координатор: следующий этап — analysis
- Агент статистической аналитики завершил работу
- Координатор: следующий этап — expert
- ТЕСТ: эксперт отклонил результаты анализа
- Координатор отправил результаты на повторный анализ. Попытка 2.
- Координатор: следующий этап — analysis
- Агент статистической аналитики завершил работу
- Координатор: следующий этап — expert
- ТЕСТ: эксперт отклонил результаты анализа
- Координатор: следующий этап — finish

Вызовов эксперта: 3
Повторных попыток: 2
Финальный этап: finish

Проверка защиты от бесконечного цикла пройдена.


### Вывод по тестированию

Проверены три сценария: обнаружение пропуска в исходных данных, повторный анализ после искусственно удалённого показателя и остановка графа после достижения лимита попыток. В последнем тесте эксперт трижды отклонил результат, координатор выполнил две повторные попытки и завершил граф. Защита от бесконечного цикла подтверждена на уровне полного тестового графа.

## 15. Сравнение одноагентной и мультиагентной реализаций

Для сопоставления используется один запрос — анализ динамики пожаров в Республике Карелия за 2020–2025 годы.



In [88]:
def single_agent_baseline(task: str, data: pd.DataFrame) -> dict:
    region_data = data[
        data["region"].str.contains("Карелия", case=False, na=False)
    ].copy()

    yearly = region_data.groupby("year")["fires"].sum().to_dict()
    years = sorted(yearly)
    first_year, last_year = years[0], years[-1]
    first_fires, last_fires = int(yearly[first_year]), int(yearly[last_year])
    difference = last_fires - first_fires
    percent = round(abs(difference) / first_fires * 100, 2)
    total_deaths = int(region_data["deaths"].sum())
    corr = round(region_data["fires"].corr(region_data["deaths"]), 3)

    prompt = f'''
Ты эксперт по пожарной безопасности.
Корреляция между пожарами и погибшими равна {corr}.
Доказывает ли она причинно-следственную связь?
Ответь одним словом: ДА или НЕТ.
'''
    response = llm.invoke(prompt)
    verdict = response.content.strip().upper()

    return {
        "first_year": first_year,
        "last_year": last_year,
        "first_fires": first_fires,
        "last_fires": last_fires,
        "difference": difference,
        "percent": percent,
        "total_deaths": total_deaths,
        "correlation": corr,
        "llm_verdict": verdict
    }


comparison_query = (
    "Проанализируй динамику пожаров в Республике Карелия "
    "за 2020–2025 годы."
)
single_result = single_agent_baseline(comparison_query, df_clean)
multi_result = graph.invoke(
    make_initial_state(comparison_query),
    config={"recursion_limit": 30}
)

print("ОДНОАГЕНТНАЯ РЕАЛИЗАЦИЯ:")
display(pd.Series(single_result, name="Значение").to_frame())
print("\nМУЛЬТИАГЕНТНАЯ СИСТЕМА:")
print(multi_result["final_result"])

ОДНОАГЕНТНАЯ РЕАЛИЗАЦИЯ:


,Значение
first_year,2020
last_year,2025
first_fires,1696
last_fires,1334
difference,-362
percent,21.34
total_deaths,258
correlation,0.787
llm_verdict,НЕТ.



МУЛЬТИАГЕНТНАЯ СИСТЕМА:
Анализ завершён.
За период 2020–2025 количество пожаров снизилось на 21.34% (-362 случаев) в регионе Республика Карелия.
В начале периода: 1696; в конце: 1334.
Всего погибших: 258.
Корреляция пожаров и погибших: 0.787.
Ограничение интерпретации подтверждено: корреляция не доказывает причинно-следственную связь.
Проверка Ollama: НЕТ.


## 16. Прототип командной строки

Функция позволяет отправлять системе произвольные текстовые запросы. Она не запускается автоматически, чтобы не блокировать выполнение ноутбука.

In [89]:
def run_cli():
    print("=" * 60)
    print("МУЛЬТИАГЕНТНАЯ СИСТЕМА АНАЛИЗА ПОЖАРНОЙ БЕЗОПАСНОСТИ")
    print("=" * 60)
    print("Примеры запросов:")
    print("- Проанализируй динамику пожаров в Республике Карелия за 2020–2025 годы.")
    print("- Сравни общее количество пожаров в России в 2020 и 2025 годах.")
    print("Для выхода введите: выход")

    while True:
        task = input("\nВаш запрос: ").strip()
        if task.lower() in {"выход", "exit", "quit"}:
            print("Работа системы завершена.")
            break
        if not task:
            print("Введите непустой запрос.")
            continue

        try:
            result = graph.invoke(
                make_initial_state(task),
                config={"recursion_limit": 30}
            )
            print("\nИТОГ:")
            print(result["final_result"])
            print("\nЖУРНАЛ:")
            for message in result["messages"]:
                print("-", message)
            print("\nПовторных попыток:", result["retry_count"])
        except Exception as error:
            print("Не удалось выполнить запрос:", error)




In [90]:
run_cli()

МУЛЬТИАГЕНТНАЯ СИСТЕМА АНАЛИЗА ПОЖАРНОЙ БЕЗОПАСНОСТИ
Примеры запросов:
- Проанализируй динамику пожаров в Республике Карелия за 2020–2025 годы.
- Сравни общее количество пожаров в России в 2020 и 2025 годах.
Для выхода введите: выход

Ваш запрос: Проанализируй динамику пожаров в Республике Карелия за 2020–2025 годы.

ИТОГ:
Анализ завершён.
За период 2020–2025 количество пожаров снизилось на 21.34% (-362 случаев) в регионе Республика Карелия.
В начале периода: 1696; в конце: 1334.
Всего погибших: 258.
Корреляция пожаров и погибших: 0.787.
Ограничение интерпретации подтверждено: корреляция не доказывает причинно-следственную связь.
Проверка Ollama: НЕТ.

ЖУРНАЛ:
- Координатор: следующий этап — quality
- Агент проверки данных завершил работу
- Координатор: следующий этап — analysis
- Агент статистической аналитики завершил работу
- Координатор: следующий этап — expert
- Агент-эксперт получил оценку Ollama и сформировал проверяемый вывод
- Эксперт одобрил результаты анализа
- Координатор: 